# 10 · Explainability

**Amaç:** Permutation importance + ağaç importance + SHAP'la global ve örnek-bazlı açıklama. Reason-code mapping kurulur (API'nin `reason_codes` alanını besler).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

from src.data.loader import load_validated

import joblib, json
from pathlib import Path
import shap
from src.data.loader import load_validated
from src.features.instant import add_derived
from src.features.historical import add_label_free_aggregates
from src.models.split import time_based_split
from src.models.train import _feature_columns

In [ ]:
df, _ = load_validated()
df = add_derived(df)
df = add_label_free_aggregates(df)
split = time_based_split(df)
num, cat = _feature_columns(df, drop_demographic=True)
feat = num + cat
model = joblib.load('../artifacts/models/hist_gbm__demographic_free.joblib')

## Permutation importance

In [ ]:
from sklearn.inspection import permutation_importance
sample = split.test.sample(min(20000, len(split.test)), random_state=0)
pi = permutation_importance(model, sample[feat], sample['IsFraudTransaction'], n_repeats=3, random_state=0, scoring='average_precision', n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=feat).sort_values(ascending=False)
imp.head(20)

## SHAP — global

In [ ]:
pre = model.named_steps['pre']
clf = model.named_steps['clf']
Xs = pre.transform(sample[feat])
fn = pre.get_feature_names_out()
expl = shap.TreeExplainer(clf)
sv = expl(Xs[:2000])
shap.summary_plot(sv, features=Xs[:2000], feature_names=fn, max_display=15)

## SHAP — örnek-bazlı (TP/FP/FN)

In [ ]:
proba = model.predict_proba(sample[feat])[:,1]
sample = sample.assign(proba=proba)
high = sample[(sample['proba']>0.8) & (sample['IsFraudTransaction']==1)].head(1)
fp = sample[(sample['proba']>0.8) & (sample['IsFraudTransaction']==0)].head(1)
fn_case = sample[(sample['proba']<0.2) & (sample['IsFraudTransaction']==1)].head(1)
print('TP example:'); print(high.iloc[0][['proba','IsFraudTransaction']]) if len(high) else print('-')
print('FP example:'); print(fp.iloc[0][['proba','IsFraudTransaction']]) if len(fp) else print('-')
print('FN example:'); print(fn_case.iloc[0][['proba','IsFraudTransaction']]) if len(fn_case) else print('-')

## Sonuç
- Top feature'lar permutation + SHAP'la tutarlı.
- Reason-code mapping `src/api/predict.py` içinde SHAP TreeExplainer ile üretiliyor.